# FunctionObject3D

A 3D shape defined by a Python function: anything that has a membership test but no primitive.

`FunctionObject3D(func, lower_bound, upper_bound)` calls `func((x, y, z))` for each point and
needs a bounding box. It has no level set and no volume: crossings and distances are found by
sampling its membership. Prefer a primitive where one fits (a torus is `Torus`, a ring
electrode `Tube`).

In [ ]:
from simgeometry import FunctionObject3D, Grid


def rounded_cube(coords):
    x, y, z = coords
    return x**4 + y**4 + z**4 <= 1.0


shape = FunctionObject3D(
    rounded_cube, lower_bound=(-1.0, -1.0, -1.0), upper_bound=(1.0, 1.0, 1.0)
)
print("corner of the cube inside:", shape.contains((0.9, 0.9, 0.9)))
print("level set:", shape.has_level_set)
assert shape.contains((0.8, 0.0, 0.0)) and not shape.contains((0.9, 0.9, 0.9))

## Volume from a node mask

Without a `volume()`, count the nodes inside: the superellipsoid $x^4 + y^4 + z^4 \le 1$ has
the volume $8\,\Gamma(5/4)^3 / \Gamma(7/4) \approx 6.48$. The surface below is also drawn from
sampled membership, which leaves faint terraces.

In [ ]:
from math import gamma

n = 41
grid = Grid(3, (-1.0, -1.0, -1.0), (1.0, 1.0, 1.0), (n, n, n))
estimate = shape.node_mask(grid).sum() * (2.0 / (n - 1)) ** 3
exact = 8 * gamma(5 / 4) ** 3 / gamma(7 / 4)
print(f"node count: {estimate:.3f}, exact: {exact:.3f}")
assert abs(estimate / exact - 1) < 0.05

In [ ]:
import plasma_plots
import pyvista as pv

pv.OFF_SCREEN = True  # render without a window, e.g. on a docs server

plotter = shape.show_3d(
    backend="pyvista", display=False, resolution=96, show_bounds=False
)
plasma_plots.scene(
    plotter, alt="A cube with rounded edges, the superellipsoid of order four"
)